In [84]:

import duckdb
import pandas
import time

duckdb_path = "data/test1.duckdb"

_SCHEMA = (
    'channel_id BIGINT, "index" DOUBLE, '
    "value UNION("
    "DT_DOUBLE DOUBLE, "
    "DT_FLOAT FLOAT, "
    "DT_SHORT SHORT, "
    "DT_LONG INT, "
    "DT_LONGLONG LONG, "
    "DT_STRING VARCHAR, "
    "DT_BOOLEAN BOOLEAN, "
    "DT_BYTE UTINYINT, "
    "DT_COMPLEX STRUCT(re FLOAT, im FLOAT), "
    "DT_DCOMPLEX STRUCT(re DOUBLE, im DOUBLE), "
    "DT_BYTESTR BLOB, "
    "DT_DATE VARCHAR"
    "), "
    "flag SHORT"
)

with duckdb.connect(duckdb_path, read_only=False) as con:
    con.execute(f"CREATE TABLE IF NOT EXISTS sm42 ({_SCHEMA})")
    con.execute("""CREATE TYPE IF NOT EXISTS exd_value_t AS UNION(
        DT_DOUBLE DOUBLE,
        DT_FLOAT FLOAT,
        DT_SHORT SHORT,
        DT_LONG INT,
        DT_LONGLONG LONG,
        DT_STRING VARCHAR,
        DT_BOOLEAN BOOLEAN,
        DT_BYTE UTINYINT,
        DT_COMPLEX STRUCT(re FLOAT, im FLOAT),
        DT_DCOMPLEX STRUCT(re DOUBLE, im DOUBLE),
        DT_BYTESTR BLOB,
        DT_DATE VARCHAR
    )
    """)

    params: list[Any] = []
    for i in range(1):
        params.extend([[i, i, 'DT_DOUBLE', 12.3, 15]])
        params.extend([[i, i, 'DT_STRING', 'test1', 15]])
        params.extend([[i, i, 'DT_LONG', 12, 15]])
        params.extend([[i, i, 'DT_COMPLEX', { 're': 1.0, 'im': 2.0 }, 15]])
        params.extend([[i, i, 'DT_DCOMPLEX', { 're': 2.1, 'im': 4.2 }, 15]])

    df = pandas.DataFrame(params, columns=['channel_id', 'index', 'dtype', 'value', 'flag'])

    #df = pandas.DataFrame([[1,2, 'DT_DOUBLE', 12.3, 15],[3,4, 'DT_STRING', "test", 15]], columns=['channel_id', 'index', 'dtype', 'value', 'flag'])
    #df.info()

    data = con.query('select channel_id, index, union_tag(value), value, flag from sm42')
    print(data)
    for i in range(10):
        started_at = time.perf_counter()

        data = con.execute("""INSERT INTO sm42 select channel_id, index, CASE
            WHEN dtype = 'DT_LONG' THEN CAST(union_value(DT_LONG := CAST(value AS INTEGER)) as exd_value_t)
            WHEN dtype = 'DT_DOUBLE' THEN CAST(union_value(DT_DOUBLE := CAST(value AS DOUBLE)) as exd_value_t)
            WHEN dtype = 'DT_FLOAT' THEN CAST(union_value(DT_FLOAT := CAST(value AS FLOAT)) as exd_value_t)
            WHEN dtype = 'DT_STRING' THEN CAST(union_value(DT_STRING := CAST(value AS VARCHAR)) as exd_value_t)
            WHEN dtype = 'DT_COMPLEX' THEN CAST(union_value(DT_COMPLEX := CAST(value AS STRUCT(re FLOAT, im FLOAT))) as exd_value_t)
            WHEN dtype = 'DT_DCOMPLEX' THEN CAST(union_value(DT_DCOMPLEX := CAST(value AS STRUCT(re DOUBLE, im DOUBLE))) as exd_value_t)
            ELSE CAST(union_value(DT_STRING := '') as exd_value_t)
            END, flag from df""")
        print(time.perf_counter() - started_at)

    data = con.query('select count(*) from sm42')
    print(data)
    data = con.query('select channel_id, index, value, union_tag(value), flag from sm42')
    print(data)

┌────────────┬────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────┐
│ channel_id │ index  │                                                                       union_tag("value")                                                                        │                                                                                                                                    value                                                                                                                                    │ flag  │
│   int64    │ double │ enum('dt_double', 'dt_float', 'dt_short', 'dt_lo

In [18]:
import pandas

params: list[Any] = []
for i in range(1000):
    params.extend([[i, i, 12.3, 15]])

df = pandas.DataFrame(params, columns=['channel_id', 'index', 'value', 'flag'])
print(df)


     channel_id  index  value  flag
0             0      0   12.3    15
1             1      1   12.3    15
2             2      2   12.3    15
3             3      3   12.3    15
4             4      4   12.3    15
..          ...    ...    ...   ...
995         995    995   12.3    15
996         996    996   12.3    15
997         997    997   12.3    15
998         998    998   12.3    15
999         999    999   12.3    15

[1000 rows x 4 columns]
